# 추가 실습 — 감쇠 진동 뉴런의 이산화: 오일러 vs 정확해(ZOH)

**Spiking Neural Networks 쿡북 with Python** · 자기주도 확장 실습

0장에서 LIF 미분방정식을 **오일러 방법**으로 풀었다. 그런데 한 가지 의문이 남았다.
6장의 2차 뉴런처럼 상태가 여럿이고 **진동(resonate)**하는 뉴런이라면, 오일러 근사가
그 진동을 제대로 따라갈 수 있을까? 시간 간격 `dt`를 크게 잡으면 무슨 일이 벌어질까?

이 노트북은 **resonate-and-fire(R&F) 뉴런의 역치하 동역학**을 감쇠 진동자 미분방정식으로
세우고, 이를 **두 가지 방법**으로 이산화해 비교한다.

- **① 전진 오일러(forward Euler)** — 매 스텝 기울기로 직진하는 근사
- **② 정확해(exact / ZOH)** — 선형이면 해가 닫힌 형태로 존재. 행렬지수로 한 스텝을 정확히 점프

### 확인하려는 것
- 오일러가 **감쇠해야 할 진동을 오히려 증폭**시키는 지점이 있는가?
- 그 경계가 되는 **임계 `dt`**는 얼마인가? 이론값과 실험값이 맞는가?


## 1. 임포트

In [ ]:
!pip install -q koreanize-matplotlib
import numpy as np
from scipy.linalg import expm          # 행렬지수 exp(At): 정확해의 핵심
import matplotlib.pyplot as plt
import koreanize_matplotlib            # 그래프 한글 폰트


## 2. R&F 뉴런의 역치하 동역학 = 감쇠 진동자

resonate-and-fire 뉴런은 이름처럼 막전위가 **공명(진동)**한다. 역치하 상태를 복소수
`z = u + i·v`로 두면 다음을 따른다.

$$\frac{dz}{dt} = (b + i\omega)\,z$$

- `b < 0` : 감쇠율. 음수이므로 진폭은 **시간이 지나며 줄어들어야** 정상이다.
- `ω`     : 각진동수. 진동을 만든다.

실수 상태 `x = [u, v]`로 풀어 쓰면 선형 시스템 `dx/dt = A·x`가 된다.

$$A = \begin{bmatrix} b & -\omega \\ \omega & b \end{bmatrix},\qquad \text{고유값} = b \pm i\omega$$


In [ ]:
b = -2.0      # 감쇠율 (<0 이면 진폭이 줄어야 정상)
w = 40.0      # 각진동수 (rad/s)
A = np.array([[b, -w],
              [w,  b]])
x0 = np.array([1.0, 0.0])   # 초기 상태
T  = 1.0                    # 총 관찰 시간 (s)
print("고유값:", np.linalg.eigvals(A))   # b±iw 확인


## 3. 두 가지 이산화

연속 방정식 `dx/dt = A·x`를 컴퓨터로 풀려면 시간을 `dt`로 쪼개 한 스텝씩 전진한다.

**① 전진 오일러** — 지금 기울기로 `dt`만큼 직진:
$$x_{n+1} = x_n + dt\,(A x_n) = (I + dt\,A)\,x_n$$

**② 정확해(ZOH)** — 선형이므로 한 스텝의 정확한 전이행렬은 행렬지수:
$$x_{n+1} = e^{A\,dt}\,x_n$$

두 방법의 차이는 딱 **전이행렬 하나**뿐이다: `(I + dt·A)` vs `expm(A·dt)`.


In [ ]:
def simulate(method, dt, T, x0):
    n = int(T / dt)
    X = np.zeros((n, 2))
    x = x0.copy()
    # 전이행렬만 다르다
    M = (np.eye(2) + dt * A) if method == 'euler' else expm(A * dt)
    for i in range(n):
        X[i] = x
        x = M @ x
    t = np.arange(n) * dt
    return t, X


## 4. 궤적 비교 — 같은 dt, 완전히 다른 운명

`dt = 0.005s`로 둘 다 돌려 막전위 `u(t)`를 겹쳐 그린다.
정답은 **감쇠(진폭이 0으로 줄어듦)**여야 한다. 오일러는 어떻게 되는지 보자.


In [ ]:
dt = 0.005
te, Xe = simulate('euler', dt, T, x0)
tx, Xx = simulate('exact', dt, T, x0)

plt.figure(figsize=(10, 4))
plt.plot(tx, Xx[:, 0], 'b',  lw=2,   label='정확해 (ZOH)')
plt.plot(te, Xe[:, 0], 'r--', lw=1.5, label='전진 오일러')
plt.axhline(0, color='gray', lw=0.5)
plt.title(f'막전위 u(t) — dt={dt}s (정답은 감쇠)')
plt.xlabel('시간 (s)'); plt.ylabel('u'); plt.legend()
plt.ylim(-8, 8)
plt.show()

print("오일러 마지막 진폭:", np.hypot(*Xe[-1]))
print("정확해 마지막 진폭:", np.hypot(*Xx[-1]))


## 5. dt를 키우며 — 언제 오일러가 무너지는가

오일러의 한 스텝 증폭계수는 `|1 + dt·λ|` (λ = b + iω)이다. 이 값이 **1을 넘으면**
매 스텝 진폭이 커져 발산한다. 여러 `dt`에 대해 이론적 증폭계수와 실제 마지막 진폭을 비교한다.


In [ ]:
print(f"{'dt':>7} | {'|1+dtλ|':>9} | {'오일러 끝진폭':>14} | {'정확해 끝진폭':>12} | 판정")
print("-"*70)
for dt in [0.001, 0.005, 0.01, 0.02, 0.05]:
    mag = abs(1 + dt*(b + 1j*w))
    _, Xe = simulate('euler', dt, T, x0)
    _, Xx = simulate('exact', dt, T, x0)
    ae, ax_ = np.hypot(*Xe[-1]), np.hypot(*Xx[-1])
    verdict = "발산" if mag > 1 else "감쇠"
    print(f"{dt:7.3f} | {mag:9.4f} | {ae:14.3f} | {ax_:12.4f} | {verdict}")


## 6. 임계 dt — 이론 vs 실험

증폭계수가 정확히 1이 되는 경계:
$$(1 + dt\,b)^2 + (dt\,\omega)^2 = 1 \;\Rightarrow\; dt^* = \frac{-2b}{b^2+\omega^2}$$

이 이론값과, 실제로 오일러가 감쇠→발산으로 뒤집히는 지점을 겹쳐 본다.


In [ ]:
dt_star = -2*b / (b**2 + w**2)
print("이론 임계 dt* =", dt_star)

dts = np.linspace(0.0005, 0.02, 300)
mags = np.abs(1 + dts*(b + 1j*w))

plt.figure(figsize=(8, 4))
plt.plot(dts, mags, 'k', lw=2, label='오일러 증폭계수 |1+dtλ|')
plt.axhline(1, color='r', ls='--', label='안정 경계 (=1)')
plt.axvline(dt_star, color='g', ls=':', label=f'이론 임계 dt*≈{dt_star:.4f}')
plt.xlabel('dt (s)'); plt.ylabel('증폭계수'); plt.legend()
plt.title('dt가 커지면 오일러는 안정 경계를 넘는다')
plt.show()


## 7. [네 차례] 직접 탐구

아래는 스스로 확인해 볼 것들이다. 값을 바꿔 돌려 보고, **관찰한 것을 직접 메모**해 두자.
(자소서·면접에서 말할 수 있는 건 네가 직접 본 것뿐이다.)

1. **`w`(진동수)를 키우면** 임계 `dt*`는 어떻게 변할까? 위 식으로 예측하고, 실험으로 확인해 보자.
2. **`b`를 0에 가깝게(약한 감쇠)** 두면 오일러는 더 쉽게 발산할까, 아닐까?
3. 정확해가 **dt를 아무리 키워도** 안정한 이유는? (힌트: `expm(A·dt)`의 고유값 크기 `exp(b·dt)`)
4. (심화) 여기에 **임계값·리셋**을 붙여 실제 R&F 발화까지 구현하면, 두 방법의 발화 패턴이 달라질까?


In [ ]:
# 예: w를 바꿔가며 이론 임계 dt* 확인
for w_try in [20.0, 40.0, 80.0]:
    dt_star = -2*b / (b**2 + w_try**2)
    print(f"w={w_try:5.1f} → 임계 dt* = {dt_star:.5f}")

# 여기에 네 실험을 이어서 작성해 보자.


## 정리 — 왜 이게 중요한가

- 오일러는 **감쇠 진동을 증폭 진동으로 질적으로 뒤바꿀** 수 있다. 단순한 수치 오차가 아니라
  **동역학의 방향 자체가 틀리는** 문제다.
- 선형 뉴런(LIF·R&F)은 **정확해가 닫힌 형태로 존재**하므로, 행렬지수(ZOH) 이산화를 쓰면
  `dt`를 키워도 안정하다.
- 실제 뉴로모픽 하드웨어(Loihi 2)와 최근 R&F 계열 연구(BRF 등)가 안정성을 위해
  exact/ZOH 계열 이산화를 채택하는 이유가 바로 이것이다.

**"어떻게 이산화하느냐"가 뉴런이 제대로 작동하느냐를 좌우한다.**
